# 04 Matching And Feature Recovery

This notebook treats **ML2R as the canonical paper-style shrinkage metric**. Raw norm ratios are not used here to
define shrinkage; they are only upstream diagnostics. The notebook therefore computes MMCS, ML2R, shrinkage gaps,
and grouped matching variants explicitly.


In [1]:
from pathlib import Path
import sys

if (Path.cwd() / 'koko_notebooks').exists():
    REPO_ROOT = Path.cwd().resolve()
else:
    REPO_ROOT = Path.cwd().resolve().parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from koko_notebooks.shrinkage_analysis.shrinkage_analysis_common import (
    OUTPUTS_DIR,
    PLOTS_DIR,
    RESULTS_DIR,
    batched_expected_masked_weight,
    batched_expected_train_mask_weight,
    collect_ci_outputs,
    component_strengths,
    discover_exp07_analysis_jsons,
    ensure_outputs_dir,
    exhaustive_binary_probe_batch,
    latest_result_per_run,
    layer_weight_metric_row,
    load_component_model_for_checkpoint,
    save_dataframe,
    save_json,
    sampled_probe_batch,
    select_consistent_replicate,
    singleton_probe_batch,
)
from koko_notebooks.shrinkage_analysis.publication_plots import (
    ARCH_COLORS,
    LAYER_COLORS,
    architecture_comparison_plot,
    heatmap,
    histogram_triptych,
    line_plot_by_group,
    line_plot_by_layer,
    multi_metric_panel_by_group,
    multi_metric_panel_by_layer,
    parse_vector_column,
    save_figure,
    setup_publication_style,
    singular_value_trajectory_plot,
)

ensure_outputs_dir()
setup_publication_style()
OUTPUTS_DIR


/root/spd_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/workspace/spd/important_outputs/shrinkage_analysis_rep1')

In [2]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from spd.utils.linear_sum_assignment import linear_sum_assignment

CONSISTENT_REPLICATE = 1

manifest_df = latest_result_per_run(discover_exp07_analysis_jsons())
manifest_df = select_consistent_replicate(manifest_df, CONSISTENT_REPLICATE)
manifest_df = manifest_df.sort_values(['depth', 'architecture', 'replicate', 'run_name']).reset_index(drop=True)

SELECTED_DEPTHS = [2, 3, 4, 5, 6]
SELECTED_ARCHITECTURES = ['tied', 'untied']
DEVICE = 'cpu'
LAYER_ORDER = list(LAYER_COLORS.keys())
SHRINKAGE_MMCS_THRESHOLD = 0.95
SHRINKAGE_ML2R_THRESHOLDS = [0.95, 0.90, 0.80]

selected_manifest_df = manifest_df[
    manifest_df['depth'].isin(SELECTED_DEPTHS) & manifest_df['architecture'].isin(SELECTED_ARCHITECTURES)
].copy()
selected_manifest_df[['run_name', 'depth', 'architecture', 'replicate']]


,run_name,depth,architecture,replicate
0,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1
1,exp_07_tms_5_2_2layer_untied_rep1,2,untied,1
2,exp_07_tms_5_2_3layer_tied_rep1,3,tied,1
3,exp_07_tms_5_2_3layer_untied_rep1,3,untied,1
4,exp_07_tms_5_2_4layer_tied_rep1,4,tied,1
5,exp_07_tms_5_2_4layer_untied_rep1,4,untied,1
6,exp_07_tms_5_2_5layer_tied_rep1,5,tied,1
7,exp_07_tms_5_2_5layer_untied_rep1,5,untied,1
8,exp_07_tms_5_2_6layer_tied_rep1,6,tied,1
9,exp_07_tms_5_2_6layer_untied_rep1,6,untied,1


In [3]:
def layer_matching_rows(component_model, run_row, checkpoint_step: int) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    eps = 1e-12
    for layer_name, components in component_model.components.items():
        target_weight = component_model.target_weight(layer_name).detach()
        target_columns = target_weight.T
        component_column_vectors = torch.einsum('ic,co->cio', components.V.detach(), components.U.detach())

        component_norm = component_column_vectors.norm(dim=-1, keepdim=True).clamp_min(eps)
        target_norm = target_columns.norm(dim=-1, keepdim=True).clamp_min(eps)
        component_unit = component_column_vectors / component_norm
        target_unit = target_columns / target_norm
        cosine_sim = torch.einsum('cio,io->ci', component_unit, target_unit)
        max_cos, max_idx = cosine_sim.max(dim=0)
        matched_component_columns = component_column_vectors[max_idx, torch.arange(target_columns.shape[0])]
        l2_ratio = matched_component_columns.norm(dim=-1) / target_columns.norm(dim=-1).clamp_min(eps)

        cost_matrix = (-cosine_sim.detach().cpu().numpy()).T
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        assignment_score = cosine_sim[col_ind, row_ind].mean().item()

        grouped_scores: dict[str, float] = {}
        for k in [1, 3, 5]:
            chosen = cosine_sim.topk(min(k, cosine_sim.shape[0]), dim=0).indices
            grouped = torch.stack([component_column_vectors[chosen[:, j], j].sum(dim=0) for j in range(target_columns.shape[0])], dim=0)
            grouped_ratio = grouped.norm(dim=-1) / target_columns.norm(dim=-1).clamp_min(eps)
            grouped_scores[f'grouped_top{k}_ml2r'] = float(grouped_ratio.mean().item())

        mmcs_value = float(max_cos.mean().item())
        ml2r_value = float(l2_ratio.mean().item())
        row = {
            'run_name': run_row['run_name'],
            'depth': int(run_row['depth']),
            'architecture': run_row['architecture'],
            'replicate': int(run_row['replicate']),
            'checkpoint_step': checkpoint_step,
            'layer_name': layer_name,
            'mmcs': mmcs_value,
            'ml2r': ml2r_value,
            'ml2r_shrinkage_gap': float(1.0 - ml2r_value),
            'assignment_cosine_mean': float(assignment_score),
            'per_feature_mmcs': max_cos.tolist(),
            'per_feature_ml2r': l2_ratio.tolist(),
            'high_mmcs_flag': float(mmcs_value >= SHRINKAGE_MMCS_THRESHOLD),
        }
        for threshold in SHRINKAGE_ML2R_THRESHOLDS:
            threshold_label = f'{threshold:.2f}'.replace('.', 'p')
            row[f'paper_shrinkage_flag_{threshold_label}'] = float(
                (mmcs_value >= SHRINKAGE_MMCS_THRESHOLD) and (ml2r_value < threshold)
            )
        row.update(grouped_scores)
        rows.append(row)
    return rows

matching_rows: list[dict[str, object]] = []
for _, manifest_row in selected_manifest_df.iterrows():
    spd_run_dir = Path(manifest_row['spd_run_dir'])
    for step in tqdm(manifest_row['checkpoint_steps'], desc=manifest_row['run_name']):
        component_model, _target_model, _config = load_component_model_for_checkpoint(spd_run_dir=spd_run_dir, step=int(step), device=DEVICE)
        matching_rows.extend(layer_matching_rows(component_model, manifest_row, int(step)))

matching_df = pd.DataFrame(matching_rows)
matching_df.head()


exp_07_tms_5_2_2layer_tied_rep1:   0%|          | 0/8 [00:00<?, ?it/s]

exp_07_tms_5_2_2layer_tied_rep1:  12%|█▎        | 1/8 [00:00<00:00,  7.30it/s]

exp_07_tms_5_2_2layer_tied_rep1:  25%|██▌       | 2/8 [00:00<00:00,  8.60it/s]

exp_07_tms_5_2_2layer_tied_rep1:  50%|█████     | 4/8 [00:00<00:00, 12.08it/s]

exp_07_tms_5_2_2layer_tied_rep1:  75%|███████▌  | 6/8 [00:00<00:00, 12.77it/s]

...

exp_07_tms_5_2_6layer_untied_rep1:  38%|███▊      | 3/8 [00:00<00:00, 10.20it/s]

exp_07_tms_5_2_6layer_untied_rep1:  62%|██████▎   | 5/8 [00:00<00:00, 10.06it/s]

exp_07_tms_5_2_6layer_untied_rep1:  88%|████████▊ | 7/8 [00:00<00:00, 10.07it/s]

exp_07_tms_5_2_6layer_untied_rep1: 100%|██████████| 8/8 [00:00<00:00, 10.09it/s]

,run_name,depth,architecture,replicate,checkpoint_step,layer_name,mmcs,ml2r,ml2r_shrinkage_gap,assignment_cosine_mean,per_feature_mmcs,per_feature_ml2r,high_mmcs_flag,paper_shrinkage_flag_0p95,paper_shrinkage_flag_0p90,paper_shrinkage_flag_0p80,grouped_top1_ml2r,grouped_top3_ml2r,grouped_top5_ml2r
0,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,5000,linear1,0.999916,0.988787,0.011213,0.999916,"[0.9999973773956299, 0.9998489022254944, 0.999...","[0.9881733059883118, 0.997978925704956, 0.9821...",1.0,0.0,0.0,0.0,0.988787,1.003681,1.030144
1,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,5000,linear2,0.637963,0.620066,0.379934,0.637963,"[0.6404476165771484, 0.6354787945747375]","[0.6217114329338074, 0.6184206008911133]",0.0,0.0,0.0,0.0,0.620066,0.967876,1.012244
2,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,10000,linear1,0.999940,0.988412,0.011588,0.999940,"[0.9999969601631165, 0.999928891658783, 0.9999...","[0.987207293510437, 0.9932093620300293, 0.9858...",1.0,0.0,0.0,0.0,0.988412,1.025965,1.032723
3,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,10000,linear2,0.638914,0.620222,0.379778,0.638914,"[0.6401190757751465, 0.637709379196167]","[0.6194198727607727, 0.6210234761238098]",0.0,0.0,0.0,0.0,0.620222,0.896138,1.017755
4,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,15000,linear1,0.999946,0.991345,0.008655,0.999946,"[0.9999998807907104, 0.9999125599861145, 0.999...","[0.9902422428131104, 0.9960888028144836, 0.985...",1.0,0.0,0.0,0.0,0.991345,1.034771,1.034208


In [4]:
matching_csv = save_dataframe(matching_df, 'csv/matching_metrics.csv')
matching_csv


PosixPath('/workspace/spd/important_outputs/shrinkage_analysis_rep1/csv/matching_metrics.csv')

In [5]:
matching_mean_df = (
    matching_df.groupby(['depth', 'architecture', 'checkpoint_step', 'layer_name'], as_index=False)
    .agg(
        mmcs=('mmcs', 'mean'),
        ml2r=('ml2r', 'mean'),
        ml2r_shrinkage_gap=('ml2r_shrinkage_gap', 'mean'),
        assignment_cosine_mean=('assignment_cosine_mean', 'mean'),
        grouped_top1_ml2r=('grouped_top1_ml2r', 'mean'),
        grouped_top3_ml2r=('grouped_top3_ml2r', 'mean'),
        grouped_top5_ml2r=('grouped_top5_ml2r', 'mean'),
        paper_shrinkage_flag_0p95=('paper_shrinkage_flag_0p95', 'mean'),
        paper_shrinkage_flag_0p90=('paper_shrinkage_flag_0p90', 'mean'),
    )
)

plot_manifest = {}
for (depth, architecture), plot_df in matching_mean_df.groupby(['depth', 'architecture'], sort=True):
    plot_manifest[f'matching_panel_depth{depth}_{architecture}'] = multi_metric_panel_by_layer(
        df=plot_df,
        x_col='checkpoint_step',
        y_cols=['mmcs', 'ml2r', 'ml2r_shrinkage_gap'],
        titles=['MMCS', 'ML2R', 'ML2R shrinkage gap = 1 - ML2R'],
        subdir='matching',
        stem=f'matching_panel_depth{depth}_{architecture}',
        hline_at_one=False,
    )
    plot_manifest[f'grouped_matching_depth{depth}_{architecture}'] = multi_metric_panel_by_layer(
        df=plot_df,
        x_col='checkpoint_step',
        y_cols=['assignment_cosine_mean', 'grouped_top3_ml2r', 'paper_shrinkage_flag_0p95'],
        titles=['Assignment cosine mean', 'Grouped top-3 ML2R', 'Paper shrinkage flag rate (<0.95)'],
        subdir='matching',
        stem=f'grouped_matching_depth{depth}_{architecture}',
        hline_at_one=False,
    )
len(plot_manifest)


20

In [6]:
representative_runs_df = selected_manifest_df.sort_values(['depth', 'architecture', 'replicate', 'run_name']).groupby(['depth', 'architecture'], as_index=False).first()

for _, rep_row in representative_runs_df.iterrows():
    rep_df = matching_df[matching_df['run_name'] == rep_row['run_name']].copy()
    for layer_name in [layer for layer in LAYER_ORDER if layer in set(rep_df['layer_name'])]:
        layer_df = rep_df[rep_df['layer_name'] == layer_name].sort_values('checkpoint_step')
        feature_matrix = parse_vector_column(layer_df['per_feature_ml2r'])
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7.5, 4.4), constrained_layout=True)
        x = layer_df['checkpoint_step'].to_numpy()
        for feature_idx in range(feature_matrix.shape[1]):
            ax.plot(x, feature_matrix[:, feature_idx], marker='o', linewidth=1.8, label=f'feature {feature_idx + 1}')
        ax.axhline(1.0, color='#444444', linestyle='--', linewidth=1)
        ax.set_xlabel('Checkpoint')
        ax.set_ylabel('Matched norm ratio (per feature)')
        ax.set_title(f'{rep_row["run_name"]} | {layer_name} per-feature ML2R')
        ax.legend(frameon=False, ncol=2)
        plot_manifest[f'per_feature_ml2r_{rep_row["run_name"]}_{layer_name}'] = save_figure(fig, subdir='matching', stem=f'per_feature_ml2r_{rep_row["run_name"]}_{layer_name}'.replace('.', '_'))
len(plot_manifest)


60

In [7]:
final_matching_df = matching_df.sort_values('checkpoint_step').groupby(['run_name', 'layer_name'], as_index=False).tail(1)
final_matching_mean_df = (
    final_matching_df.groupby(['architecture', 'depth', 'layer_name'], as_index=False)
    .agg(
        mmcs=('mmcs', 'mean'),
        ml2r=('ml2r', 'mean'),
        ml2r_shrinkage_gap=('ml2r_shrinkage_gap', 'mean'),
        paper_shrinkage_flag_0p95=('paper_shrinkage_flag_0p95', 'mean'),
    )
)

for architecture in ['tied', 'untied']:
    arch_df = final_matching_mean_df[final_matching_mean_df['architecture'] == architecture].copy()
    ordered_layers = [layer for layer in LAYER_ORDER if layer in set(arch_df['layer_name'])]
    mmcs_matrix = arch_df.pivot(index='depth', columns='layer_name', values='mmcs').reindex(columns=ordered_layers).sort_index()
    ml2r_matrix = arch_df.pivot(index='depth', columns='layer_name', values='ml2r').reindex(columns=ordered_layers).sort_index()
    gap_matrix = arch_df.pivot(index='depth', columns='layer_name', values='ml2r_shrinkage_gap').reindex(columns=ordered_layers).sort_index()
    plot_manifest[f'mmcs_heatmap_{architecture}'] = heatmap(
        matrix=mmcs_matrix.to_numpy(),
        row_labels=[str(idx) for idx in mmcs_matrix.index],
        col_labels=list(mmcs_matrix.columns),
        title=f'Final MMCS | {architecture}',
        colorbar_label='MMCS',
        subdir='matching',
        stem=f'mmcs_heatmap_{architecture}',
        vmin=0.5,
        vmax=1.0,
        annotate=True,
    )
    plot_manifest[f'ml2r_heatmap_{architecture}'] = heatmap(
        matrix=ml2r_matrix.to_numpy(),
        row_labels=[str(idx) for idx in ml2r_matrix.index],
        col_labels=list(ml2r_matrix.columns),
        title=f'Final ML2R | {architecture}',
        colorbar_label='ML2R',
        subdir='matching',
        stem=f'ml2r_heatmap_{architecture}',
        vmin=0.3,
        vmax=1.1,
        annotate=True,
    )
    plot_manifest[f'shrinkage_gap_heatmap_{architecture}'] = heatmap(
        matrix=gap_matrix.to_numpy(),
        row_labels=[str(idx) for idx in gap_matrix.index],
        col_labels=list(gap_matrix.columns),
        title=f'Final ML2R shrinkage gap | {architecture}',
        colorbar_label='1 - ML2R',
        subdir='matching',
        stem=f'shrinkage_gap_heatmap_{architecture}',
        vmin=0.0,
        vmax=max(0.05, float(gap_matrix.to_numpy().max())),
        cmap='OrRd',
        annotate=True,
    )

save_json(plot_manifest, 'plots/matching/manifest.json')
plot_manifest


{'matching_panel_depth2_tied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/matching/matching_panel_depth2_tied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/matching/matching_panel_depth2_tied.pdf'},
 'grouped_matching_depth2_tied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/matching/grouped_matching_depth2_tied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/matching/grouped_matching_depth2_tied.pdf'},
 'matching_panel_depth2_untied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/matching/matching_panel_depth2_untied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/matching/matching_panel_depth2_untied.pdf'},
 'grouped_matching_depth2_untied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/matching/grouped_matching_depth2_untied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage